In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

d:\Capstone_project_mlops\MLOPS_Capstone_Project\mlops_capstone_project\.venv\Lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
243,I love Umberto Lenzi's cop movies -- ROME ARME...,negative
132,"""Ambushed"" is no ordinary action flick. It's m...",negative
516,"Back in 1994, I had a really lengthy vacation ...",positive
803,"As a South African, living in South Africa aga...",negative
772,"I wanted to watch this, to get a inside look a...",positive


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
243,love umberto lenzi s cop movie rome armed teet...,negative
132,ambushed ordinary action flick much bad ordina...,negative
516,back really lengthy vacation around fourth jul...,positive
803,south african living south africa year stay uk...,negative
772,wanted watch this get inside look show told st...,positive


In [5]:
df['sentiment'].value_counts()

sentiment
negative    266
positive    234
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
243,love umberto lenzi s cop movie rome armed teet...,0
132,ambushed ordinary action flick much bad ordina...,0
516,back really lengthy vacation around fourth jul...,1
803,south african living south africa year stay uk...,0
772,wanted watch this get inside look show told st...,1


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
import os
from pathlib import Path

import dagshub
from dotenv import load_dotenv

load_dotenv(Path("../.env").resolve())

CONFIG = {
    "repo_owner": os.getenv("DAGSHUB_REPO_OWNER"),
    "repo_name": os.getenv("DAGSHUB_REPO_NAME"),
    "mlflow_tracking_uri": os.getenv(
        "MLFLOW_TRACKING_URI",
        f"https://dagshub.com/{os.getenv('DAGSHUB_REPO_OWNER')}/{os.getenv('DAGSHUB_REPO_NAME')}.mlflow",
    ),
    "experiment_name": "Logistic Regression Baseline",
}

if not os.getenv("DAGSHUB_USER_TOKEN"):
    raise ValueError(
        "DAGSHUB_USER_TOKEN is not set. Add it to mlops_capstone_project/.env "
        "or run: dagshub login"
    )

if not CONFIG["repo_owner"] or not CONFIG["repo_name"]:
    raise ValueError(
        "DAGSHUB_REPO_OWNER and DAGSHUB_REPO_NAME must be set in mlops_capstone_project/.env"
    )

dagshub.init(repo_owner=CONFIG["repo_owner"], repo_name=CONFIG["repo_name"], mlflow=True)
mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
mlflow.set_experiment(CONFIG["experiment_name"])

Initialized MLflow to track repo "sheezarafique266/MLOPS_Capstone_Project"

Repository sheezarafique266/MLOPS_Capstone_Project initialized!

2026/06/12 16:36:37 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/89225a7ff1bb43998a6f6032afc2f496', creation_time=1781264197043, experiment_id='0', last_update_time=1781264197043, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [15]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-12 16:40:29,014 - INFO - Starting MLflow run...
2026-06-12 16:40:30,473 - INFO - Logging preprocessing parameters...
2026-06-12 16:40:31,644 - INFO - Initializing Logistic Regression model...
2026-06-12 16:40:31,645 - INFO - Fitting the model...
2026-06-12 16:40:31,658 - INFO - Model training complete.
2026-06-12 16:40:31,659 - INFO - Logging model parameters...
2026-06-12 16:40:32,052 - INFO - Making predictions...
2026-06-12 16:40:32,054 - INFO - Calculating evaluation metrics...
2026-06-12 16:40:32,059 - INFO - Logging evaluation metrics...
2026-06-12 16:40:33,642 - INFO - Saving and logging the model...
2026/06/12 16:40:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/06/12 16:40:39 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2026-06-12 16:40:39,758 - INFO